In [1]:
# -*- coding: utf-8 -*-
import torch
import torch.nn as nn
from torchvision.models.vgg import vgg11#有现成的VGG组件了
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

In [8]:
# 根据环境参数选取对应运行平台：cpu or gpu
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print('Running Device:', device)

# 对获取的图像数据做ToTensor()变换和归一化
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

# 利用torchvision提供的CIFAR10数据集类，实例化训练集和测试集提取类
trainset = torchvision.datasets.CIFAR10(root='../data', train=True, download=False, transform=transform)
testset = torchvision.datasets.CIFAR10(root='../data', train=False, download=False, transform=transform)

# 利用torch提供的DataLoader, 实例化训练集DataLoader 和 测试集DataLoader
Batch_size = 8
trainLoader = torch.utils.data.DataLoader(trainset, batch_size=Batch_size, shuffle=True, num_workers=2)
testLoader = torch.utils.data.DataLoader(testset, batch_size=Batch_size, shuffle=False, num_workers=2)

# CIFAR10 类别内容
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')



Running Device: cpu


RuntimeError: Dataset not found or corrupted. You can use download=True to download it

In [ ]:
# 卷积神经网络实例化
net = vgg11(num_classes = 10, pretrained=False, progress=True).to(device)
print(net)

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (11): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (12): ReLU(inplace=True)
    (13): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (14): ReLU(inplace=True)
    (15): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
 

In [ ]:
# 实例化损失函数和SGD优化子
epochs = 3
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)
print("start")
# 卷积神经网络训练
for epoch in range(epochs):
    running_loss = 0.0

    for i, data in enumerate(trainLoader, 0):
        #if i > 10:
        #    break
        inputs, labels = data
        # 将数据迁移到device中，如device为GPU，则将数据从CPU迁移到GPU；如device为CPU，则将数据从CPU迁移到CPU（即不作移动）
        inputs, labels = inputs.to(device), labels.to(device)

        # 清空参数的梯度值
        optimizer.zero_grad()

        # 前馈+反馈+优化
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # 统计训练损失
        running_loss += loss.item()

        if i % 2000 == 19:    # 每2000 mini-batches，打印一次
            print('[%d, %5d] loss: %.3f' %
                    (epoch + 1, i + 1, running_loss / 2000))
            running_loss = 0.0

print('Finished Training')

start
[1,    20] loss: 0.023
[1,    40] loss: 0.023
[1,    60] loss: 0.023
[1,    80] loss: 0.023
[1,   100] loss: 0.023
[1,   120] loss: 0.023
[1,   140] loss: 0.023
[1,   160] loss: 0.023
[1,   180] loss: 0.023
[1,   200] loss: 0.023
[1,   220] loss: 0.023
[1,   240] loss: 0.023
[1,   260] loss: 0.023
[1,   280] loss: 0.023
[1,   300] loss: 0.023
[1,   320] loss: 0.023
[1,   340] loss: 0.023
[1,   360] loss: 0.023
[1,   380] loss: 0.023
[1,   400] loss: 0.023
[1,   420] loss: 0.023
[1,   440] loss: 0.023
[1,   460] loss: 0.022
[1,   480] loss: 0.023
[1,   500] loss: 0.022
[1,   520] loss: 0.022
[1,   540] loss: 0.022
[1,   560] loss: 0.022
[1,   580] loss: 0.022
[1,   600] loss: 0.022
[1,   620] loss: 0.021
[1,   640] loss: 0.021
[1,   660] loss: 0.021
[1,   680] loss: 0.022
[1,   700] loss: 0.021
[1,   720] loss: 0.021
[1,   740] loss: 0.022
[1,   760] loss: 0.020
[1,   780] loss: 0.020
[1,   800] loss: 0.020
[1,   820] loss: 0.021
[1,   840] loss: 0.020
[1,   860] loss: 0.020
[1,  

In [ ]:
# 卷积神经网络测试
correct = 0
total = 0
with torch.no_grad():     # 测试阶段，停止梯度计算
    for data in testLoader:
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device) #

        # 前馈获得网络预测结果
        outputs = net(inputs)
        # 在预测结果中，获得预测值最大化的类别id
        _, predicted = torch.max(outputs.data, 1)

        # 统计样本个数
        total += labels.size(0)
        # 统计正确预测样本个数
        correct += (predicted == labels).sum().item()

print('Accuracy of the network on the 10000 test inputs: %d %%' % (100 * correct / total))


Accuracy of the network on the 10000 test inputs: 9 %


In [ ]:
class_correct = list(0. for i in range(10))
class_total = list(0. for i in range(10))
print("start")

with torch.no_grad():
    print("start")
    for data in testLoader:
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)

        # print(inputs, labels)
        # 前馈获得网络预测结果
        outputs = net(inputs)
        # 在预测结果中，获得预测值最大化的类别id
        _, predicted = torch.max(outputs.data, 1)

        # 对于batch内每一个样本，获取其预测是否正确
        c = (predicted == labels)#.squeeze()
        #print(c)
        #print((predicted == labels).shape)
        # 对于batch内样本，统计每个类别的预测正确率
        for i in range(Batch_size):
            label = labels[i]
            class_correct[label] += c[i].item()
            class_total[label] += 1

for i in range(10):
    print('Accuracy of %5s : %2d %%' % (classes[i], 100 * class_correct[i] / class_total[i]))

start
start
Accuracy of plane : 14 %
Accuracy of   car :  3 %
Accuracy of  bird :  1 %
Accuracy of   cat : 41 %
Accuracy of  deer :  1 %
Accuracy of   dog :  0 %
Accuracy of  frog : 29 %
Accuracy of horse :  1 %
Accuracy of  ship :  0 %
Accuracy of truck :  9 %
